# Setting Up and Invoking Custom Guardrails with IBM watsonx.governance

This notebook demonstrates how to create and manage **Custom Guardrails** using the IBM watsonx.governance Guardrails Manager API.


## Prerequisites

Before running this notebook, you need:

1. **IBM watsonx.governance Instance**
   - Create one at: https://cloud.ibm.com/catalog/services/watsonxgovernance
   - After creation, get the Service Instance ID from the URL (last UUID)

2. **IBM Cloud API Key**
   - Create at: https://cloud.ibm.com/iam/apikeys
   - Click "Create" and save the API key securely

3. **Inventory ID**
   - UUID format identifier for your inventory
   - Obtained from your watsonx.governance setup see the [documentation](https://www.ibm.com/docs/en/waasfgm?topic=cloud-setting-up-default-inventory)

4. **Custom Detector Endpoint** (for creating custom detectors)
   - URL of your external detection service
   - API key for authentication

## Install Dependencies



In [1]:
import requests
import json



## Configuration



In [ ]:
# Required Configuration replace with your actual values
SERVICE_INSTANCE_ID = "<EDIT THIS>"   
IBM_CLOUD_APIKEY = "<EDIT THIS>"    
INVENTORY_ID = "<EDIT THIS>"

### Supported Regions:
# us-south (Dallas), eu-de (Frankfurt), au-syd (Sydney), ca-tor (Toronto), jp-tok (Tokyo)
WATSONX_REGION = "us-south"


In [ ]:
# API Endpoints
IAM_URL = "https://iam.cloud.ibm.com/oidc/token"
BASE_URL = "https://aiopenscale.cloud.ibm.com"
DETECTORS_URL = f"{BASE_URL}/guardrails_manager/v2/detectors"
POLICIES_URL = f"{BASE_URL}/guardrails_manager/v2/policies"
ENFORCE_URL = f"{BASE_URL}/guardrails_manager/v2/enforce"

## Authentication
Create a token using the service api key mentioned above


In [3]:
def get_access_token():
    """Generate IBM Cloud access token from API key."""
    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Accept": "application/json"
    }
    data = {
        "grant_type": "urn:ibm:params:oauth:grant-type:apikey",
        "apikey": IBM_CLOUD_APIKEY
    }
    
    try:
        response = requests.post(IAM_URL, data=data, headers=headers)
        status = response.status_code

        if status == 200:
            print("Token obtained successfully")
            return response.json().get("access_token")

        elif status in (400, 401, 403):
            err = response.json()
            raise Exception(f"Auth error {status}: {err.get('error_description', response.text)}")

        elif status == 429:
            raise Exception("Rate limit exceeded.")
        else:
            raise Exception(f"Unexpected status {status}: {response.text}")
    except Exception as e:
        raise Exception(f"Authentication failed: {e}")


token = get_access_token()



Token obtained successfully


In [ ]:
# Setup common headers for all API requests
headers = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {token}',
    'X-Governance-Instance-Id': SERVICE_INSTANCE_ID
}

## List Available Custom Detectors


In [4]:
params = {'inventory_id': INVENTORY_ID}
response = requests.get(DETECTORS_URL, headers=headers, params=params)

if response.status_code == 200:
    detectors_data = response.json()
    detectors = detectors_data.get("custom_detectors", [])# To view the built in detectors change "custom_detectors" to "detectors"
    if not detectors:
        print("No detectors found.")
    else:
        print(json.dumps(detectors, indent=2, ensure_ascii=False))
else:
    print(f"Error listing detectors: {response.status_code}")
    print(json.dumps(response.json(), indent=2))



[
  {
    "entity": {
      "id": "b1ef6541-a6e0-4b02-b4bc-3134bf9feace",
      "inventory_id": "709626aa-aad7-4bd2-8c8f-e9f76bac3227",
      "name": "custom_profanity_detectorv3",
      "description": "Custom detector for profanity detection using external service",
      "url": "https://your-custom-detector-endpoint.com/detect",
      "actions": [
        "block",
        "mask"
      ],
      "metadata": {
        "secret_id": "019bae29-9b5a-78b8-bf30-a0e60523bd32"
      },
      "detector_properties": [
        {
          "name": "threshold",
          "type": "number",
          "min": 0,
          "max": 1,
          "default_value": 0.7
        },
        {
          "name": "strict_mode",
          "type": "bool",
          "default_value": false
        }
      ],
      "parameters": [
        {
          "name": "language",
          "is_optional": true
        }
      ],
      "status": {
        "state": "active"
      }
    },
    "metadata": {
      "id": "b1ef6541-a6e0-

## Create a Custom Detector



Custom detectors allow you to integrate external detection services into watsonx.governance.





### Detector Configuration

- **name**: Unique identifier for your detector
- **url**: Endpoint of your detection service
- **api_key**: Customers custom detector authentication key
- **direction**: Where to apply (`input`, `output`, or both)
- **detector_properties**: Configurable parameters (threshold, mode, etc.)
- **parameters**: Optional runtime parameters

In [5]:
### Example: Custom Profanity Detector
custom_detector_payload = {
    "name": "custom_profanity_detectorv4", # This must be unique, if you have used a name before you can not reuse it
    "description": "Custom detector for profanity detection using external service",
    "url": "https://your-custom-detector-endpoint.com/detect", # NOTE: Replace with your actual detector endpoint and API key
    "api_key": "your-detector-api-key", # NOTE: Replace with your actual detector endpoint and API key
    "direction": ["input", "output"],
    "detector_properties": [
        {
            "name": "threshold",
            "type": "number",
            "min": 0.0,
            "max": 1.0,
            "default_value": 0.7
        },
        {
            "name": "strict_mode",
            "type": "bool",
            "default_value": False
        }
    ],
    "parameters": [
        {
            "name": "language",
            "is_optional": True
        }
    ]
}

# Create the detector
params = {'inventory_id': INVENTORY_ID}
response = requests.post(
    DETECTORS_URL,
    headers=headers,
    params=params,
    json=custom_detector_payload
)

if response.status_code == 201:
    detector_response = response.json()
    detector_id = detector_response['metadata']['id']
    detectors_name = detector_response['entity']['name']
    # Store the detector_id, and detector_name for later use
    CUSTOM_DETECTOR_ID = detector_id
    CUSTOM_DETECTOR_NAME = detector_response['entity']['name']
    print("Custom detector created successfully")
    print(json.dumps(detector_response, indent=2, ensure_ascii=False))
else:
    print(f"Error creating detector: {response.status_code}")
    print(json.dumps(response.json(), indent=2))    


Custom detector created successfully
{
  "entity": {
    "id": "b2d8bd59-7823-4fe8-b50c-5df34a61927b",
    "inventory_id": "709626aa-aad7-4bd2-8c8f-e9f76bac3227",
    "name": "custom_profanity_detectorv4",
    "description": "Custom detector for profanity detection using external service",
    "url": "https://your-custom-detector-endpoint.com/detect",
    "actions": [
      "block",
      "mask"
    ],
    "metadata": {
      "secret_id": "019bb194-b2f0-7995-b9a2-e67fbcdbba20"
    },
    "detector_properties": [
      {
        "name": "threshold",
        "type": "number",
        "min": 0,
        "max": 1,
        "default_value": 0.7
      },
      {
        "name": "strict_mode",
        "type": "bool",
        "default_value": false
      }
    ],
    "parameters": [
      {
        "name": "language",
        "is_optional": true
      }
    ],
    "status": {
      "state": "active"
    }
  },
  "metadata": {
    "id": "b2d8bd59-7823-4fe8-b50c-5df34a61927b",
    "created_at": "2

## Create a Policy with Custom Detector
Policies define:
- Which detectors to use
- What actions to take (block or mask)
- Different rules for input vs output
- Configuration for each detector

### Actions
- **block**: Prevent content from being processed
- **mask**: Redact detected content with a character (e.g., `*`)

### Policy Status
- **publish**: Active policy (enforced)
- **draft**: Inactive policy (not enforced)

Note: When using custom detectors in policies, you MUST always include a special property called `custom_detector_id` that references the detector's unique identifier. This is **in addition to** all the properties you defined during detector creation.


In [ ]:
# Policy configuration
policy_payload = {
    "name": "custom_profanity_detectorv4", # This must be unique, if you have used a name before you can not reuse it
    "description": "Policy using custom profanity detector with blocking and masking actions",
    "block_message": "Content blocked due to policy violation. Please revise your input.",
    "mask_character": "*",
    "input": [
        {
            "detector": CUSTOM_DETECTOR_NAME, 
            "action": "block",
            "detector_properties": [
                {"name": "custom_detector_id", "value": CUSTOM_DETECTOR_ID},  # When using costume detectors, this property is a must  
                {"name": "threshold", "value": "0.8"}, # Note how we are using the same property that we defined at the step before when we create a custom detector
                {"name": "strict_mode", "value": "true"} # Note how we are using the same property that we defined at the step before when we create a custom detector
            ]
        }
    ],
    "output": [
        {
            "detector": CUSTOM_DETECTOR_NAME,
            "action": "mask",
            "detector_properties": [
                {"name": "custom_detector_id", "value": CUSTOM_DETECTOR_ID},  # When using costume detectors, this property is a must                                
                {"name": "threshold", "value": "0.7"}, # Note how we are using the same property that we defined at the step before when we create a custom detector
                {"name": "strict_mode", "value": "false"}  # Note how we are using the same property that we defined at the step before when create a custom detector
            ]
        }
    ],
    "policy_status": "publish",
    "tags": ["custom", "profanity", "content-safety"]
}

# Create the policy
params = {'inventory_id': INVENTORY_ID}
response = requests.post(
    POLICIES_URL,
    headers=headers,
    params=params,
    json=policy_payload
)

if response.status_code == 201:
    policy_response = response.json()
    policy_id = policy_response['metadata']['id']
    print("Policy created successfully")
    POLICY_ID = policy_id
else:
    print(f"Error creating policy: {response.status_code}")
    print(json.dumps(response.json(), indent=2))



Policy created successfully


## List and Retrieve Policies

View all policies and get details about specific ones.


In [20]:
# List all published policies
params = {
    'inventory_id': INVENTORY_ID,
    'policytype': 'publish'  # Options: 'publish', 'draft', 'false' (all)
}
response = requests.get(POLICIES_URL, headers=headers, params=params)

if response.status_code == 200:
    policies = response.json()
    policy_list = policies.get('policies', []) 
    print(f"Found {len(policy_list)} published policies")
    print(json.dumps(policies, indent=2, ensure_ascii=False))    
    
else:
    print(f"Error listing policies: {response.status_code}")
    print(json.dumps(response.json(), indent=2))    


Found 8 published policies
{
  "policies": [
    {
      "entity": {
        "inventory_id": "709626aa-aad7-4bd2-8c8f-e9f76bac3227",
        "id": "",
        "name": "custom_profanity_detectorv4",
        "description": "Policy using custom profanity detector with blocking and masking actions",
        "input": [
          {
            "detector": "custom_profanity_detectorv4"
          }
        ],
        "output": [
          {
            "detector": "custom_profanity_detectorv4"
          }
        ],
        "tags": [
          "custom",
          "profanity",
          "content-safety"
        ],
        "status": {
          "state": "active"
        }
      },
      "metadata": {
        "id": "87a853ee-3d2b-4149-964e-48cac5bdd4fa",
        "created_at": "2026-01-12T09:41:34Z",
        "created_by": "IBMid-697000GE0W",
        "modified_at": "2026-01-12T09:41:34Z",
        "modified_by": "IBMid-697000GE0W"
      }
    },
    {
      "entity": {
        "inventory_id": "70962

## Enforce Policy



In [ ]:
if 'POLICY_ID' in locals():
    test_text = "This is a test message "
    enforce_payload = {
        "text": test_text,
        "direction": "input",
        "detectors_properties": {
            "custom_profanity_detectorv4": {  # This must match the detector name that we created at step "Create a Costume detector"
                "custom_detector_id": CUSTOM_DETECTOR_ID,
                "threshold": "0.8",
                "strict_mode": "true"
            }
        }
    }    
    params = {'inventory_id': INVENTORY_ID}
    response = requests.post(
        f"{ENFORCE_URL}/{POLICY_ID}", 
        headers=headers, 
        params=params, 
        json=enforce_payload
    )
    if response.status_code == 200:
        result = response.json()
        print("Enforcement Result:")
        print(f"Overall Status: {result['entity']['status']['overall']}")
        print(f"Total Detectors: {result['entity']['status']['summary']['total_detectors']}")
        print(f"Succeeded: {result['entity']['status']['summary']['succeeded']}")
        print(f"Failed: {result['entity']['status']['summary']['failed']}")
    else:
        print(f"Error: {response.status_code}")
        print(json.dumps(response.json(), indent=2))
else:
    print("Create a policy first")



Enforcement Result:
Overall Status: failure
Total Detectors: 1
Succeeded: 0
Failed: 1


## Update an existing Policy 
This cell demonstrates how to update an existing policy using PUT request
You can modify detector configurations, actions, thresholds, or any policy settings

In [ ]:
# You can change fields such as detector settings, actions, block_message, mask_character
updated_policy_payload = {
    "name": "custom_profanity_detectorv3", # you can't change the name, note that the name is the same name we of the policy we created at step "Create a Policy with Custom Detector"
    "description": "Updated policy with modified threshold values and settings",
    "block_message": "Content blocked due to updated policy rules. Please revise your input.",
    "mask_character": "#",
    "input": [
        {
            "detector": CUSTOM_DETECTOR_NAME, 
            "action": "block",# Note: You can change from "block" to "mask"
            "detector_properties": [
                {"name": "custom_detector_id", "value": CUSTOM_DETECTOR_ID},                
                {"name": "threshold", "value": "0.9"},
                {"name": "strict_mode", "value": "true"}
            ]
        }
    ],
    "output": [
        {
            "detector": CUSTOM_DETECTOR_NAME,
            "action": "mask", # Note: You can change from "mask" to "block"
            "detector_properties": [
                {"name": "custom_detector_id", "value": CUSTOM_DETECTOR_ID},                                
                {"name": "threshold", "value": "0.6"},
                {"name": "strict_mode", "value": "false"}
            ]
        }
    ],
    "policy_status": "publish",
    "tags": ["custom", "profanity", "content-safety", "updated_v2"]  # You can add a tag to point out that it was updated
}

if 'POLICY_ID' not in locals():
    print("No policy found to modify. Please create a policy first.")
else:
    params = {'inventory_id': INVENTORY_ID}
    response = requests.put(
        f"{POLICIES_URL}/{POLICY_ID}",  # Note: When we want to update a specific policy we use PUT and provide specific policy ID
        headers=headers,
        params=params,
        json=updated_policy_payload
    )

    if response.status_code == 200:
        policy_response = response.json()
        print("Policy updated successfully")
        print(f"Policy ID: {policy_response['metadata']['id']}")
        print(f"Policy Name: {policy_response['entity']['name']}")
    else:
        print(f"Error updating policy: {response.status_code}")
        print(json.dumps(response.json(), indent=2))



Policy updated successfully
Policy ID: 1711b269-fce9-4216-919b-2d80475a11d4
Policy Name: custom_profanity_detectorv3


## Policy Enforcement with Built-in Detectors
This section demonstrates how to enforce policies using IBM's built-in detectors.
 We'll create two separate policies:


### HAP Detection with BLOCK Action
Create a policy that uses the HAP detector to block harmful content.

In [13]:
# HAP Policy configuration
hap_policy_payload = {
    "name": "HAP Content Blocking Policy2",
    "description": "Policy to block hate, abuse, and profanity content",
    "block_message": "Your content contains inappropriate language and has been blocked. Please revise your message.",
    "input": [
        {
            "detector": "hap", 
            "action": "block",
            "detector_properties": [
                {"name": "threshold", "value": "0.75"}
            ]
        }
    ],
    "output": [
        {
            "detector": "hap",
            "action": "block",
            "detector_properties": [
                {"name": "threshold", "value": "0.75"}
            ]
        }
    ],
    "policy_status": "publish",
    "tags": ["hap", "content-safety", "blocking"]
}


params = {'inventory_id': INVENTORY_ID}
response = requests.post(
    POLICIES_URL,
    headers=headers,
    params=params,
    json=hap_policy_payload
)

if response.status_code == 201:
    hap_policy_response = response.json()
    HAP_POLICY_ID = hap_policy_response['metadata']['id']
    print("HAP Policy created successfully")
    print(f"Policy ID: {HAP_POLICY_ID}")
    print(f"Policy Name: {hap_policy_response['entity']['name']}")
else:
    print(f"Error creating HAP policy: {response.status_code}")
    print(json.dumps(response.json(), indent=2))


HAP Policy created successfully
Policy ID: f03d6dba-33a5-4408-9a4e-3808e3cdadbf
Policy Name: HAP Content Blocking Policy2


 ### Enforce HAP Policy - BLOCK Example

In [ ]:
if 'HAP_POLICY_ID' in locals():
    
    hap_test_text = "I hate people from the moon"
    hap_enforce_payload = {
        "text": hap_test_text,
        "direction": "input",
        "detectors_properties": {
            "hap": {
                "threshold": "0.75"
            }
        }
    }
    
    print(f"Test Text: {hap_test_text}")
    params = {'inventory_id': INVENTORY_ID}
    response = requests.post(
        f"{ENFORCE_URL}/{HAP_POLICY_ID}",
        headers=headers,
        params=params,
        json=hap_enforce_payload
    )
    
    if response.status_code == 200:
        result = response.json()
        
        was_blocked = result['entity']['text'] != hap_test_text
        print("Enforcement Result:")
        print(f"Status: {result['entity']['status']['overall']}")
        print(f"Detectors Run: {result['entity']['status']['summary']['total_detectors']}")
        print(f"Detectors Succeeded: {result['entity']['status']['summary']['succeeded']}")
        
        if was_blocked:
            print(f"Content was BLOCKED")
            print(f"Block Message: {result['entity']['text']}")
        else:
            print(f"Content was ALLOWED")
            print(f"Processed Text: {result['entity']['text']}")


Test Text: I hate people from the moon
Enforcement Result:
Status: success
Detectors Run: 1
Detectors Succeeded: 1
Content was BLOCKED
Block Message: Your content contains inappropriate language and has been blocked. Please revise your message.


### PII Detection with MASK Action
Create a policy that uses the PII detector to mask sensitive information.


In [15]:
pii_policy_payload = {
    "name": "PII Data Masking Policy2",
    "description": "Policy to mask personally identifiable information",
    "mask_character": "*",
    "input": [
        {
            "detector": "pii",
            "action": "mask",
            "detector_properties": [
                {"name": "threshold", "value": "0.5"}
            ]
        }
    ],
    "output": [
        {
            "detector": "pii",
            "action": "mask",
            "detector_properties": [
                {"name": "threshold", "value": "0.5"}
            ]
        }
    ],
    "policy_status": "publish",
    "tags": ["pii", "data-privacy", "masking"]
}

params = {'inventory_id': INVENTORY_ID}
response = requests.post(
    POLICIES_URL,
    headers=headers,
    params=params,
    json=pii_policy_payload
)

if response.status_code == 201:
    pii_policy_response = response.json()
    PII_POLICY_ID = pii_policy_response['metadata']['id']
    print("PII Policy created successfully")
    print(f"Policy ID: {PII_POLICY_ID}")
    print(f"Policy Name: {pii_policy_response['entity']['name']}")
else:
    print(f"Error creating PII policy: {response.status_code}")
    print(json.dumps(response.json(), indent=2))


PII Policy created successfully
Policy ID: 9de8ed84-d8ff-48f9-aab9-1cc13c1cf5e1
Policy Name: PII Data Masking Policy2


### Enforce PII Policy - MASK Example


In [16]:
if 'PII_POLICY_ID' in locals():

    pii_test_text = "My email is john.doe@example.com and my phone number is 555-123-4567. I live at 123 Main Street."
    pii_enforce_payload = {
        "text": pii_test_text,
        "direction": "output",
        "detectors_properties": {
            "pii": {
                "threshold": "0.5"
            }
        }
    }
    
    print(f"Test Text: {pii_test_text}")
    params = {'inventory_id': INVENTORY_ID}
    response = requests.post(
        f"{ENFORCE_URL}/{PII_POLICY_ID}",
        headers=headers,
        params=params,
        json=pii_enforce_payload
    )
    
    if response.status_code == 200:
        result = response.json()
        returned_text = result['entity']['text']
        
        was_masked = returned_text != pii_test_text
        
        print("Enforcement Result:")
        print(f"Overall Status: {result['entity']['status']['overall']}")
        print(f"Total Detectors: {result['entity']['status']['summary']['total_detectors']}")
        print(f"Succeeded: {result['entity']['status']['summary']['succeeded']}")
        print(f"Failed: {result['entity']['status']['summary']['failed']}")
        
        if was_masked:
            print(f"PII was MASKED by the detector")
            print(f"Original Text: {pii_test_text}")
            print(f"Masked Text:   {returned_text}")
        else:
            print(f"No PII detected or masking applied")
            print(f"Returned Text: {returned_text}")
    else:
        print(f"Error enforcing policy: {response.status_code}")
        print(json.dumps(response.json(), indent=2))


Test Text: My email is john.doe@example.com and my phone number is 555-123-4567. I live at 123 Main Street.
Enforcement Result:
Overall Status: success
Total Detectors: 1
Succeeded: 1
Failed: 0
PII was MASKED by the detector
Original Text: My email is john.doe@example.com and my phone number is 555-123-4567. I live at 123 Main Street.
Masked Text:   My email is ******************** and my phone number is ************. I live at 123 Main Street.
